In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [4]:
# Simply ToTensor
# Normalization from the pytorch documentation :)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [5]:
batch_size = 64

In [6]:
# Loading the data
trainset = torchvision.datasets.CIFAR10(root = './data', train = True, download = True, transform = transform)
trainloader = DataLoader(trainset, batch_size = batch_size, shuffle = True)
testset = torchvision.datasets.CIFAR10(root = './data', train = False, download = True, transform = transform)
testloader = DataLoader(testset, batch_size = batch_size, shuffle = False)

100%|████████████████████████| 170498071/170498071 [00:48<00:00, 3525418.13it/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [7]:
# Simple CNN model 
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.convolutional_1 = nn.Conv2d(3, 32, kernel_size=3, padding=1) # (32,32,32)
        self.relu_1 = nn.ReLU()
        self.pooling_1 = nn.MaxPool2d(2, 2)  # (32,16,16)

        self.convolutional_2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # (64,16,16)
        self.relu_2 = nn.ReLU()
        self.pooling_2 = nn.MaxPool2d(2, 2)  # (64,8,8)

        self.fc_1 = nn.Linear(64 * 8 * 8, 128)
        self.relu_3 = nn.ReLU()
        self.fc_2 = nn.Linear(128, 10)  # 10 classes:
        ''' airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck'''

    def forward(self, x):
        x = self.pooling_1(self.relu_1(self.convolutional_1(x)))
        x = self.pooling_2(self.relu_2(self.convolutional_2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = self.relu_3(self.fc_1(x))
        x = self.fc_2(x)
        return x

In [14]:
learning_rate = 0.001
num_epochs = 20

In [15]:
# Init model, cost function, optimizer
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

In [16]:
# Use cuda if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SimpleCNN(
  (convolutional_1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu_1): ReLU()
  (pooling_1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (convolutional_2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu_2): ReLU()
  (pooling_2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc_1): Linear(in_features=4096, out_features=128, bias=True)
  (relu_3): ReLU()
  (fc_2): Linear(in_features=128, out_features=10, bias=True)
)

In [17]:
# practice makes perfect:

for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}')

Epoch 1, Loss: 1.2634674181871097
Epoch 2, Loss: 0.8928428656609771
Epoch 3, Loss: 0.7414107468655652
Epoch 4, Loss: 0.620645611182503
Epoch 5, Loss: 0.5083072363872967
Epoch 6, Loss: 0.4103389115589659
Epoch 7, Loss: 0.31566742785713253
Epoch 8, Loss: 0.24068600703459567
Epoch 9, Loss: 0.17762686068768543
Epoch 10, Loss: 0.13451371040157117
Epoch 11, Loss: 0.1106966795564136
Epoch 12, Loss: 0.09213556356065909
Epoch 13, Loss: 0.08814786615617135
Epoch 14, Loss: 0.07481994464233652
Epoch 15, Loss: 0.07441347335462871
Epoch 16, Loss: 0.07109975741456842
Epoch 17, Loss: 0.0633686188700111
Epoch 18, Loss: 0.06019501326962367
Epoch 19, Loss: 0.05875411117959278
Epoch 20, Loss: 0.05996178910273182


In [18]:
# model evaluation:

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy on testset: {correct / total * 100}%')

Accuracy on testset: 71.43%


In [19]:
import torchvision.models as models

In [20]:
# ResNet18 model 
class ResNet18(nn.Module):
    def __init__(self):
        super(ResNet18, self).__init__()
        self.model = models.resnet18(pretrained = False)
        self.model.fc = nn.Linear(self.model.fc.in_features, 10)

    def forward(self, x):
        return self.model(x)

In [21]:
model = ResNet18()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)
model.to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet18(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runn

In [22]:
# training resnet18

for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(trainloader)
    print(f'Epoch {epoch+1}, Loss: {avg_loss}')

Epoch 1, Loss: 1.3757239115969908
Epoch 2, Loss: 0.9813135594053342
Epoch 3, Loss: 0.8054818105514702
Epoch 4, Loss: 0.6820891256756185
Epoch 5, Loss: 0.5774307257264776
Epoch 6, Loss: 0.4805037912810245
Epoch 7, Loss: 0.400871401552654
Epoch 8, Loss: 0.31747953043035837
Epoch 9, Loss: 0.25928281599660513
Epoch 10, Loss: 0.20152540721685228
Epoch 11, Loss: 0.17386765386480504
Epoch 12, Loss: 0.14840765553228843
Epoch 13, Loss: 0.12234292301954344
Epoch 14, Loss: 0.1125619896411267
Epoch 15, Loss: 0.10846242835492734
Epoch 16, Loss: 0.09084581066687565
Epoch 17, Loss: 0.08321189687883868
Epoch 18, Loss: 0.07495657826923882
Epoch 19, Loss: 0.08197135446697969
Epoch 20, Loss: 0.07655103048940887


In [23]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy on testset: {correct / total * 100}%')

Accuracy on testset: 77.06%
